## HealthConnect Clinic — Machine Learning Problem Definition
**Week 4 / AnalystLab Africa Experience Lab**

This notebook defines the machine learning problem for reducing missed appointments at HealthConnect Clinic, based on the appointment dataset and data dictionary. This stage focuses on problem definition and initial assessment.

## 1. Problem Definition

**Business problem:** HealthConnect Clinic experiences a high rate of missed appointments (no-shows), leading to high costs, wasted appointment slots, inefficient resource allocation, and reduced patient care capacity.

**ML framing:** This is a **binary classification problem**. Predicting, at or shortly after booking time, whether a scheduled patient is likely to attend (`Attended`) or miss (`No-Show`) their appointment. A reliable prediction would allow the clinic to intervene proactively (e.g., targeted reminders, overbooking strategies) for high-risk appointments.

### Data Loading and Inspection

In [20]:
# importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
#loading dataset
df = pd.read_csv('Downloads/HealthConnect_Appointment_Data.csv')

In [22]:
# First 10 rows of the data
df.head(10)

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show
5,HC-00006,P-0523,Male,66,65+,Specialist Consultation,2/16/2026,4/14/2026,Tuesday,Morning,57,4,2,No,NaN,3.1,36.0,No-Show
6,HC-00007,P-0776,Male,58,55-64,Diagnostic Test,1/17/2025,2/16/2025,Sunday,Afternoon,30,4,1,Yes,WhatsApp,8.2,30.0,No-Show
7,HC-00008,P-0927,Female,28,25-34,Specialist Consultation,11/27/2025,1/16/2026,Friday,Afternoon,50,6,1,Yes,SMS,19.2,21.0,Attended
8,HC-00009,P-0388,Male,77,65+,Follow-up,3/9/2025,4/21/2025,Monday,Afternoon,43,3,0,Yes,SMS,10.1,27.0,No-Show
9,HC-00010,P-0371,Female,20,18-24,Follow-up,2/5/2025,2/21/2025,Friday,Afternoon,16,2,0,Yes,SMS,28.0,20.0,Attended


In [23]:
# shape of the data
df.shape

(5000, 18)

**Insights**: The data has 5000 rows and 18 columns

In [24]:
#data statistics
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   str    
 1   patient_id             5000 non-null   str    
 2   gender                 5000 non-null   str    
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   str    
 5   appointment_type       5000 non-null   str    
 6   booking_date           5000 non-null   str    
 7   appointment_date       5000 non-null   str    
 8   appointment_day        5000 non-null   str    
 9   appointment_time       5000 non-null   str    
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   str    
 14  reminder_channel       3634 non-null   str    
 15  distance_to_cli

In [25]:
df.describe()

,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


In [26]:
df.dtypes

appointment_id               str
patient_id                   str
gender                       str
age                        int64
age_group                    str
appointment_type             str
booking_date                 str
appointment_date             str
appointment_day              str
appointment_time             str
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent                str
reminder_channel             str
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome          str
dtype: object

### Data Assessment

In [27]:
# Missing values
df.isnull().sum().sort_values(ascending=False)

reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_id              0
patient_id                  0
gender                      0
booking_date                0
age                         0
age_group                   0
appointment_type            0
appointment_time            0
appointment_day             0
appointment_date            0
booking_lead_days           0
reminder_sent               0
previous_no_shows           0
previous_appointments       0
appointment_outcome         0
dtype: int64

**Finding:** Three columns contain missing values: reminder_channel (1,366), distance_to_clinic_km (90), waiting_time_minutes (60). No missing values in the demographic, booking, or the target column itself.

In [28]:
# checking for duplicated rows
print("Full row duplicates:", df.duplicated().sum())
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())
print("Duplicate patient_id (repeat patients — expected):", df['patient_id'].duplicated().sum())

Full row duplicates: 0
Duplicate appointment_id: 0
Duplicate patient_id (repeat patients — expected): 3304


**Logic**: No duplicate appointment records. The values in the `patient_id` do not reflect true duplicates. It is expected that one patient can have more than one appointment.

### Logical Consistency Checks

In [29]:
bad = df[df['previous_no_shows'] > df['previous_appointments']]
print("Rows where previous_no_shows > previous_appointments:", len(bad))

print("\nbooking_lead_days range:", df['booking_lead_days'].min(), "-", df['booking_lead_days'].max())
print("age range:", df['age'].min(), "-", df['age'].max())

Rows where previous_no_shows > previous_appointments: 0

booking_lead_days range: 0 - 60
age range: 18 - 80


**Justification**: The check hopes to identify if there are any patient that had more `previous_no_show` than previous_appointments. It helps us to check whether certain age groups have higher booking lead times or no-show rates.

### Explaining the missing values

In [30]:
print(pd.crosstab(df['reminder_sent'], df['reminder_channel'].isna()))

reminder_channel  False  True 
reminder_sent                 
No                    0   1366
Yes                3634      0


**Logic:** `reminder_channel` missing value is fully explained by `reminder_sent = No`. Every one of the 1,366 missing values corresponds exactly to a patient who was not sent a reminder. This is structural, not a data quality defect, and would not be imputed as if it were random.

In [31]:
print("Missing distance_to_clinic_km by outcome:")
print(df[df['distance_to_clinic_km'].isna()]['appointment_outcome'].value_counts())


Missing distance_to_clinic_km by outcome:
appointment_outcome
No-Show      47
Attended     39
Cancelled     4
Name: count, dtype: int64


In [32]:
print("Missing waiting_time_minutes by outcome:")
print(df[df['waiting_time_minutes'].isna()]['appointment_outcome'].value_counts())

Missing waiting_time_minutes by outcome:
appointment_outcome
No-Show      37
Attended     21
Cancelled     2
Name: count, dtype: int64


**Logic:** Missing values in `distance_to_clinic_km` and `waiting_time_minutes` is spread proportionally across outcome categories, with no concentration in one class, which is consistent with data missing at random rather than a systematic recording issue tied to the outcome.

### Proposed Target variable

In [33]:
print(df['appointment_outcome'].value_counts())
print(df['appointment_outcome'].value_counts(normalize=True).round(4))

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64
appointment_outcome
No-Show      0.4846
Attended     0.4628
Cancelled    0.0526
Name: proportion, dtype: float64


**Findings**: appointment_outcome contains three categories: No-Show (48.5%), Attended (46.3%), and Cancelled (5.3%).

**Proposed handling:** Cancelled appointments (5.3% of the data) represent a distinct patient action (an active decision communicated in advance) rather than a passive non-attendance. Since the business problem centers specifically on unplanned missed appointments, Cancelled rows are proposed to be excluded from the primary model, with the target binarized as Attended vs. No-Show.

### Potential Features & Leakage Assessment

In [36]:
# Checking waiting_time_minutes for leakage
print(df.groupby('appointment_outcome')['waiting_time_minutes'].describe())

                      count       mean        std  min   25%   50%   75%   max
appointment_outcome                                                           
Attended             2293.0  24.289141  10.946389  2.0  17.0  24.0  32.0  68.0
Cancelled             261.0  23.214559  10.557896  2.0  16.0  22.0  29.0  55.0
No-Show              2386.0  24.200754  10.814945  2.0  17.0  24.0  31.0  67.0


**Why it matters:** If a field is not known before the appointment happens, it can’t be used in a real model.   
**What this output shows:** `waiting_time_minutes` looks almost the same across Attended, Cancelled, and No-Show, so it does not separate the classes well.   
**Handling**: It should be excluded from the model because it is not reliable at prediction time and adds little useful signal.

### Checking on other candidate features. 

In [37]:
print("Mean booking_lead_days by outcome:")
print(df.groupby('appointment_outcome')['booking_lead_days'].mean().round(2))

print("\nMean previous_no_shows by outcome:")
print(df.groupby('appointment_outcome')['previous_no_shows'].mean().round(3))

print("\nNo-show rate by reminder_sent:")
print(pd.crosstab(df['reminder_sent'], df['appointment_outcome'], normalize='index').round(3))

print("\nMean distance_to_clinic_km by outcome:")
print(df.groupby('appointment_outcome')['distance_to_clinic_km'].mean().round(2))

Mean booking_lead_days by outcome:
appointment_outcome
Attended     24.52
Cancelled    29.63
No-Show      34.53
Name: booking_lead_days, dtype: float64

Mean previous_no_shows by outcome:
appointment_outcome
Attended     0.457
Cancelled    0.418
No-Show      0.641
Name: previous_no_shows, dtype: float64

No-show rate by reminder_sent:
appointment_outcome  Attended  Cancelled  No-Show
reminder_sent                                    
No                      0.427      0.059    0.514
Yes                     0.476      0.050    0.474

Mean distance_to_clinic_km by outcome:
appointment_outcome
Attended      9.67
Cancelled    10.11
No-Show      10.53
Name: distance_to_clinic_km, dtype: float64


**Logic:** This checks whether these features help explain missed appointments: booking lead time, past no-shows, reminders, and distance to clinic.

**Finding:**   
`booking_lead_days` shows a real difference. No-Show patients book, on average, 10 days further in advance (34.5 vs. 24.5 days) than those who attend. Plausible real-world driver: longer lead time gives more opportunity for circumstances to change.  
`previous_no_shows` shows the clearest behavioral signal — patients who eventually no-show average 0.64 prior no-shows vs. 0.46 for those who attend, supporting the intuitive idea that past behavior predicts future behavior.  
`reminder_sent` shows only a mild effect (51.4% no-show without reminder vs. 47.4% with); it is worth including as a feature, but not expected to be a dominant predictor on its own.  
`distance_to_clinic_km` shows a small difference (10.5 km vs. 9.7 km) — a modest candidate signal.  

### Proposed Feature List

In [38]:
candidate_features = [
    'age', 'gender', 'appointment_type', 'booking_lead_days',
    'previous_appointments', 'previous_no_shows', 'reminder_sent',
    'reminder_channel', 'distance_to_clinic_km', 'appointment_day',
    'appointment_time'
]

print(df[candidate_features].dtypes)

age                        int64
gender                       str
appointment_type             str
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent                str
reminder_channel             str
distance_to_clinic_km    float64
appointment_day              str
appointment_time             str
dtype: object


**Logic:** Some columns were removed from the feature list, as they do not add any predictive value.

`appointment_id`, `patient_id` — identifiers; no predictive value   
`booking_date`, `appointment_date` — raw dates already represented via derived fields (`booking_lead_days`, `appointment_day`); could revisit for seasonality features in a later phase  
`waiting_time_minutes` — leakage/realism concern   
`age_group` — redundant with `age`; retaining both is unnecessary. 

### Initial Modelling Approach




At this stage, no model has been trained — this section outlines the proposed direction for Week 5 onward:

- **Problem type:** Binary classification (Attended vs. No-Show)
- **Candidate baseline models:** Logistic Regression (easy to interpret) and a tree-based model such as Random Forest or Gradient Boosting (to capture more complex patterns, e.g., between `previous_no_shows` and `booking_lead_days`)
- **Preprocessing anticipated:** Encoding for categorical fields (`gender`, `appointment_type`, `appointment_day`, `appointment_time`, `reminder_sent`, `reminder_channel`); scaling for numeric fields depending on the chosen model family
- **Evaluation approach:** Given the near-balanced target (51.2%/48.8%), accuracy is a reasonable starting metric, but precision/recall and F1 should also be tracked — a false negative (predicting attendance when a patient will no-show) has different operational cost than a false positive, and this should inform metric choice going into Week 5

### Key Modelling Considerations, Assumptions, Limitations & Risks

**Assumptions:**
- Cancelled appointments are treated as a distinct patient action, separate from unplanned no-shows, for the purposes of this initial model
- Missing values in `distance_to_clinic_km` and `waiting_time_minutes` is assumed to be at random, based on their proportional spread across outcome categories

**Limitations:**
- This is a fictional, synthetic dataset — the `waiting_time_minutes` anomaly (populated for no-show patients) confirms the data does not perfectly mirror real-world clinic behavior, and findings should not be assumed to generalize to a live clinical setting without validation
- 5,000 records is a moderate sample size for a clinic-wide model; subgroup analysis (e.g., by `appointment_type`) may be limited by smaller subgroup sizes

**Risks:**
- `waiting_time_minutes` was identified as a leakage risk and excluded — but similar risks should be re-checked for any newly engineered features in later phases
- Reminder-related fields (`reminder_sent`, `reminder_channel`) may reflect clinic *response* to perceived risk (e.g., staff already flagging high-risk patients for reminders) rather than a purely independent cause — this potential circularity should be considered when interpreting feature importance later

**Dependencies:**
- Final modelling work in later weeks depends on the Data Analytics track's parallel exploration of the same dataset, and any Machine Learning Engineering track decisions about deployment format, which may influence which features are practical to use in production